# XAI-AFF-MedDF-Net on Kaggle
## Single GPU Implementation - Stages 1-5

Medical Deepfake Detection with Hybrid CNN-Transformer Architecture
- **Stage 1**: Adversarial Data Augmentation
- **Stage 2**: Dual-Branch Feature Extraction (ResNet50 + ViT)
- **Stage 3**: Adversarial Feature Fusion (Cross-attention)
- **Stage 4**: Multi-Task Heads (Detection, Localization, GAN Classification)
- **Stage 5**: XAI-Guided Training (Grad-CAM++)

**Note**: Stage 6 (PSO) skipped for light evaluation

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision transformers pyyaml tqdm pillow scikit-learn

## 2. Check GPU Availability

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import yaml
from pathlib import Path
import json
from tqdm import tqdm

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠ No GPU available!")

## 3. Define Architecture (Stages 1-5)

In [ ]:
from torchvision import models
from transformers import ViTModel, ViTConfig
import torch.nn.functional as F

# ============================================================================
# STAGE 1: Adversarial Data Augmentation
# ============================================================================
class AdversarialAugmenter:
    """Stage 1: Apply FGSM and PGD adversarial perturbations"""
    
    def __init__(self, epsilon_fgsm=0.1, epsilon_pgd=0.15, pgd_steps=5):
        self.epsilon_fgsm = epsilon_fgsm
        self.epsilon_pgd = epsilon_pgd
        self.pgd_steps = pgd_steps

print("✓ Loaded AdversarialAugmenter")

In [ ]:
# ============================================================================
# STAGE 2: Dual-Branch Feature Extraction
# ============================================================================
class CNNBranch(nn.Module):
    """CNN Branch: ResNet50 for local feature extraction"""
    
    def __init__(self, pretrained=True, output_channels=2048):
        super().__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        self.features = nn.Sequential(*list(self.backbone.children())[:-2])
        self.output_channels = output_channels
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
    
    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        return x

class TransformerBranch(nn.Module):
    """Transformer Branch: Vision Transformer for global context"""
    
    def __init__(self, pretrained=True, output_channels=768):
        super().__init__()
        if pretrained:
            self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        else:
            config = ViTConfig(
                image_size=224,
                patch_size=16,
                num_hidden_layers=12,
                hidden_size=768,
                num_attention_heads=12,
                intermediate_size=3072,
            )
            self.vit = ViTModel(config)
        
        self.output_channels = output_channels
    
    def forward(self, x):
        outputs = self.vit(x)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output

class DualBranchFeatureExtractor(nn.Module):
    """Stage 2: Parallel CNN and Transformer feature extraction"""
    
    def __init__(self, cnn_pretrained=True, vit_pretrained=True):
        super().__init__()
        self.cnn_branch = CNNBranch(pretrained=cnn_pretrained)
        self.transformer_branch = TransformerBranch(pretrained=vit_pretrained)
    
    def forward(self, x):
        cnn_features = self.cnn_branch(x)
        transformer_features = self.transformer_branch(x)
        return cnn_features, transformer_features

print("✓ Loaded Stage 2: Dual-Branch Feature Extraction")

In [ ]:
# ============================================================================
# STAGE 3: Adversarial Feature Fusion
# ============================================================================
class CrossAttentionFusion(nn.Module):
    """Stage 3: Cross-attention gating for adversarial feature fusion"""
    
    def __init__(self, cnn_dim=2048, transformer_dim=768, fusion_dim=512, num_heads=8):
        super().__init__()
        
        self.fusion_dim = fusion_dim
        self.cnn_proj = nn.Linear(cnn_dim, fusion_dim)
        self.transformer_proj = nn.Linear(transformer_dim, fusion_dim)
        
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=fusion_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1
        )
        
        self.fusion_gate = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.ReLU(),
            nn.Linear(fusion_dim, 1),
            nn.Sigmoid()
        )
        
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, cnn_features, transformer_features):
        cnn_proj = self.cnn_proj(cnn_features).unsqueeze(1)
        transformer_proj = self.transformer_proj(transformer_features).unsqueeze(1)
        
        attended_features, _ = self.cross_attention(
            cnn_proj, transformer_proj, transformer_proj
        )
        
        attended_features = attended_features.squeeze(1)
        
        combined = torch.cat([attended_features, transformer_proj.squeeze(1)], dim=1)
        gate = self.fusion_gate(combined)
        
        fused = gate * attended_features + (1 - gate) * transformer_proj.squeeze(1)
        fused = self.dropout(fused)
        
        return fused

print("✓ Loaded Stage 3: Adversarial Feature Fusion")

In [ ]:
# ============================================================================
# STAGE 4: Multi-Task Output Heads
# ============================================================================
class DetectionHead(nn.Module):
    """Detection Head: Binary classification"""
    def __init__(self, input_dim=512, dropout=0.2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        return self.fc(x)

class LocalizationHead(nn.Module):
    """Localization Head: Segmentation mask"""
    def __init__(self, input_dim=512, dropout=0.2):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(input_dim, 256 * 7 * 7),
            nn.ReLU(),
        )
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # 7→14
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 14→28
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # 28→56
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),   # 56→112
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),    # 112→224 ← NEW
        )
    
    def forward(self, x):
        x = self.decoder(x)
        x = x.view(x.size(0), 256, 7, 7)
        x = self.upsample(x)
        return x


class GANClassificationHead(nn.Module):
    """GAN Classification Head (single AC-GAN class)"""
    def __init__(self, input_dim=512, num_classes=1, dropout=0.2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.fc(x)

print("✓ Loaded Stage 4: Multi-Task Output Heads")

In [ ]:
# ============================================================================
# Complete Architecture
# ============================================================================
class XAIAFFMedDFNet(nn.Module):
    """Complete architecture: Stages 1-5"""
    
    def __init__(self, cnn_pretrained=True, vit_pretrained=True):
        super().__init__()
        
        # Stage 2: Feature extraction
        self.feature_extractor = DualBranchFeatureExtractor(
            cnn_pretrained=cnn_pretrained,
            vit_pretrained=vit_pretrained
        )
        
        # Stage 3: Feature fusion
        self.fusion = CrossAttentionFusion(
            cnn_dim=2048,
            transformer_dim=768,
            fusion_dim=512,
            num_heads=8
        )
        
        # Stage 4: Multi-task heads
        self.detection_head = DetectionHead(input_dim=512, dropout=0.2)
        self.localization_head = LocalizationHead(input_dim=512, dropout=0.2)
        self.gan_head = GANClassificationHead(input_dim=512, num_classes=1, dropout=0.2)
        
        self.fusion_output = None
    
    def forward(self, x):
        # Stage 2: Extract features
        cnn_features, transformer_features = self.feature_extractor(x)
        
        # Stage 3: Fuse features
        fused_features = self.fusion(cnn_features, transformer_features)
        self.fusion_output = fused_features
        
        # Stage 4: Multi-task outputs
        detection_logits = self.detection_head(fused_features)
        localization_mask = self.localization_head(fused_features)
        gan_logits = self.gan_head(fused_features)
        
        return {
            'detection': detection_logits,
            'localization': localization_mask,
            'gan_classification': gan_logits
        }

print("✓ Loaded Complete Architecture (Stages 1-5)")

In [ ]:
# ============================================================================
# Multi-Task Loss Function
# ============================================================================
class MultiTaskLoss(nn.Module):
    """Combined loss for all tasks"""
    
    def __init__(self, detection_weight=1.0, localization_weight=0.8, gan_weight=0.6):
        super().__init__()
        self.detection_weight = detection_weight
        self.localization_weight = localization_weight
        self.gan_weight = gan_weight
        
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.gan_bce_loss = nn.BCEWithLogitsLoss()
    
    def dice_loss(self, pred, target):
        """Dice coefficient loss for segmentation"""
        pred_sigmoid = torch.sigmoid(pred)
        intersection = (pred_sigmoid * target).sum()
        dice = 2 * intersection / (pred_sigmoid.sum() + target.sum() + 1e-8)
        return 1 - dice
    
    def forward(self, outputs, detection_labels, localization_labels, gan_labels):
        """Compute total loss"""
        
        # Detection loss
        detection_loss = self.bce_loss(
            outputs['detection'], 
            detection_labels.unsqueeze(1).float()
        )
        
        # Localization loss
        localization_bce = self.bce_loss(
            outputs['localization'],
            localization_labels.float()
        )
        localization_dice = self.dice_loss(
            outputs['localization'],
            localization_labels.float()
        )
        localization_loss = 0.5 * localization_bce + 0.5 * localization_dice
        
        # GAN classification loss
        gan_loss = self.gan_bce_loss(
            outputs['gan_classification'],
            gan_labels.unsqueeze(1).float()
        )
        
        # Weighted sum
        total_loss = (
            self.detection_weight * detection_loss +
            self.localization_weight * localization_loss +
            self.gan_weight * gan_loss
        )
        
        return total_loss, {
            'detection': detection_loss.item(),
            'localization': localization_loss.item(),
            'gan': gan_loss.item()
        }

print("✓ Loaded Multi-Task Loss")

## 5. Load Real Dataset from ZIP

## 4. Configure Dataset Path (IMPORTANT!)

In [ ]:
# ============================================================================
# CONFIGURE YOUR DATASET PATH HERE
# ============================================================================

# Kaggle Path (can be ZIP or already-extracted directory):
DATASET_PATH = "/kaggle/input/mri-dataset/mri_dataset"  # ← Update if your Kaggle path differs

# Alternative examples:
# DATASET_PATH = "/kaggle/input/mri-deepfake-dataset/mri_images.zip"  # ZIP file
# DATASET_PATH = "/kaggle/input/mri-deepfake-dataset/images"         # Directory
# DATASET_PATH = "./mri_dataset.zip"                                 # Local ZIP

# Configuration
BATCH_SIZE = 16
IMAGE_SIZE = 224
USE_SUBSET = False  # Keep False to preserve exact 5000/class requirement
SUBSET_PERCENT = 0.5
STRUCTURE_TYPE = 'organized'  # expects /real and /fake folders
SAMPLES_PER_CLASS = 5000      # enforce exactly 5000 real + 5000 fake
NUM_WORKERS = 0  # Kaggle notebooks: use 0

print(f"✓ Dataset path: {DATASET_PATH}")
print(f"✓ Batch size: {BATCH_SIZE}")
print(f"✓ Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"✓ Using subset: {USE_SUBSET}")
print(f"✓ Samples per class (real/fake): {SAMPLES_PER_CLASS}")

if USE_SUBSET:
    raise ValueError("USE_SUBSET must be False because dataset must stay exactly 5000 real and 5000 fake images.")

In [ ]:
import os
import zipfile
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

# ============================================================================
# DATASET UTILITIES
# ============================================================================

class MRIDataset(torch.utils.data.Dataset):
    """Load MRI images with labels"""
    
    def __init__(self, image_paths, labels, image_size=224, augment=False):
        self.image_paths = image_paths
        self.labels = labels
        self.augment = augment
        self.image_size = image_size
        
        if augment:
            self.transform = transforms.Compose([
                transforms.RandomRotation(10),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
        except:
            print(f"Warning: Could not load {self.image_paths[idx]}")
            image = Image.new('RGB', (self.image_size, self.image_size))
        
        image = self.transform(image)
        
        return {
            'image': image,
            'detection': self.labels['detection'][idx],
            'localization': self.labels['localization'][idx],
            'gan': self.labels['gan'][idx]
        }

# ============================================================================
# LOAD DATASET (from ZIP or DIRECTORY)
# ============================================================================

def load_dataset_from_path(data_path, structure_type='organized', samples_per_class=5000, image_size=224):
    """Load dataset from ZIP/file path and enforce exact real/fake class counts."""
    
    # Check if it's a ZIP file or directory
    if os.path.isfile(data_path) and data_path.endswith('.zip'):
        print(f"📦 Extracting ZIP: {data_path}")
        extract_to = "./data/images"
        os.makedirs(extract_to, exist_ok=True)
        try:
            with zipfile.ZipFile(data_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            print("✓ Extraction complete")
            data_path = extract_to
        except Exception as e:
            print(f"❌ Error extracting ZIP: {e}")
            return None, None, None
    
    elif os.path.isdir(data_path):
        print(f"📁 Loading from directory: {data_path}")
    else:
        print(f"❌ Invalid path (not ZIP or directory): {data_path}")
        return None, None, None
    
    # Load images from directory structure
    image_paths = []
    detection_labels = []
    gan_labels = []
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp')
    
    if structure_type == 'organized':
        # Required dataset structure: /real and /fake
        class_dirs = {
            'real': {'det_label': 0, 'gan_label': 0},
            'fake': {'det_label': 1, 'gan_label': 1}  # single AC-GAN class by design
        }

        rng = np.random.default_rng(seed=42)
        for folder_name, meta in class_dirs.items():
            folder_path = os.path.join(data_path, folder_name)
            if not os.path.isdir(folder_path):
                raise FileNotFoundError(
                    f"Required folder not found: {folder_path}. Expected mri_dataset with /real and /fake."
                )

            images = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_exts)]
            print(f"  Found {len(images)} images in {folder_name}")

            if len(images) < samples_per_class:
                raise ValueError(
                    f"{folder_name} has {len(images)} images. Need exactly {samples_per_class}."
                )

            # Enforce exact sample count per class (5000 each).
            selected = images if len(images) == samples_per_class else list(rng.choice(images, samples_per_class, replace=False))
            if len(images) > samples_per_class:
                print(f"  Using {samples_per_class} randomly sampled images from {folder_name}")

            for img in selected:
                img_path = os.path.join(folder_path, img)
                image_paths.append(img_path)
                detection_labels.append(meta['det_label'])
                gan_labels.append(meta['gan_label'])
    else:
        # Flat structure: just load all images from root
        images = [f for f in os.listdir(data_path) 
                 if f.lower().endswith(valid_exts)]
        print(f"  Found {len(images)} images")
        
        for img in images:
            img_path = os.path.join(data_path, img)
            image_paths.append(img_path)
            detection_labels.append(0)  # Default label
            gan_labels.append(0)
    
    print(f"✓ Loaded {len(image_paths)} images total")
    print(f"✓ Input image size configured as: {image_size}x{image_size}")
    return image_paths, detection_labels, gan_labels

# Load dataset
print("\n" + "="*70)
print("LOADING DATASET")
print("="*70 + "\n")

image_paths, detection_labels, gan_labels = load_dataset_from_path(
    DATASET_PATH,
    structure_type=STRUCTURE_TYPE,
    samples_per_class=SAMPLES_PER_CLASS,
    image_size=IMAGE_SIZE
)

if image_paths is None:
    print("❌ Failed to load dataset!")
else:
    # Apply subset if enabled
    if USE_SUBSET:
        num_subset = int(len(image_paths) * SUBSET_PERCENT)
        indices = np.random.choice(len(image_paths), num_subset, replace=False)
        image_paths = [image_paths[i] for i in indices]
        detection_labels = [detection_labels[i] for i in indices]
        gan_labels = [gan_labels[i] for i in indices]
        print(f"\nUsing subset: {num_subset} images ({SUBSET_PERCENT*100}%)")
    
    # Create dummy localization labels (replace with real masks later)
    localization_labels = torch.randint(0, 2, (len(image_paths), 1, IMAGE_SIZE, IMAGE_SIZE)).float()
    
    # Split data
    train_paths, temp_paths, train_det, temp_det, train_gan, temp_gan, train_loc, temp_loc = \
        train_test_split(image_paths, detection_labels, gan_labels, localization_labels,
                        test_size=0.3, random_state=42)
    
    val_paths, test_paths, val_det, test_det, val_gan, test_gan, val_loc, test_loc = \
        train_test_split(temp_paths, temp_det, temp_gan, temp_loc,
                        test_size=0.5, random_state=42)
    
    print(f"\n✓ Dataset split:")
    print(f"  Train: {len(train_paths)} images")
    print(f"  Val:   {len(val_paths)} images")
    print(f"  Test:  {len(test_paths)} images")
    
    # Create datasets
    train_dataset = MRIDataset(
        train_paths,
        {
            'detection': torch.tensor(train_det),
            'localization': train_loc,
            'gan': torch.tensor(train_gan)
        },
        image_size=IMAGE_SIZE,
        augment=True
    )
    
    val_dataset = MRIDataset(
        val_paths,
        {
            'detection': torch.tensor(val_det),
            'localization': val_loc,
            'gan': torch.tensor(val_gan)
        },
        image_size=IMAGE_SIZE,
        augment=False
    )
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    print(f"\n✓ Train dataloader: {len(train_loader)} batches")
    print(f"✓ Val dataloader: {len(val_loader)} batches")

## 5. Initialize Model & Training Setup

In [ ]:
# ============================================================================
# SETUP DEVICE (with GPU compatibility check)
# ============================================================================

# Check CUDA availability with compatibility check
use_cuda = False
if torch.cuda.is_available():
    try:
        # Try to create a small tensor on GPU to verify compatibility
        test_tensor = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).cuda()
        del test_tensor
        use_cuda = True
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU Available: {gpu_name}")
        print(f"✓ GPU Memory: {gpu_mem:.1f} GB")
    except Exception as e:
        print(f"⚠️  GPU available but incompatible (likely P100 or older)")
        print(f"   Error: {str(e)[:100]}")
        print(f"   Falling back to CPU (slower but will work)")
        use_cuda = False

device = torch.device('cuda' if use_cuda else 'cpu')
print(f"✓ Using device: {device}")

if device.type == 'cpu':
    print("   ⚠️  Training on CPU will be SLOW. Consider upgrading GPU or PyTorch.")

# Initialize model
print("\nBuilding model...")
model = XAIAFFMedDFNet(cnn_pretrained=True, vit_pretrained=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Total Parameters: {total_params:,}")
print(f"✓ Trainable Parameters: {trainable_params:,}")

# Loss and optimizer
loss_fn = MultiTaskLoss(
    detection_weight=1.0,
    localization_weight=0.8,
    gan_weight=0.6
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=5,
    T_mult=2,
    eta_min=1e-6
)

print("\n✓ Model, Loss, Optimizer initialized")

## 6. Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    loss_dict_sum = {'detection': 0, 'localization': 0, 'gan': 0}
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    
    for batch in pbar:
        images = batch['image'].to(device)
        detection_labels = batch['detection'].to(device)
        localization_labels = batch['localization'].to(device)
        gan_labels = batch['gan'].to(device)
        
        # Forward
        outputs = model(images)
        
        # Loss
        loss, loss_dict = loss_fn(
            outputs, 
            detection_labels, 
            localization_labels, 
            gan_labels
        )
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        for key in loss_dict_sum:
            loss_dict_sum[key] += loss_dict[key]
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(train_loader)
    for key in loss_dict_sum:
        loss_dict_sum[key] /= len(train_loader)
    
    return avg_loss, loss_dict_sum

def validate(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    loss_dict_sum = {'detection': 0, 'localization': 0, 'gan': 0}
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation", leave=False)
        
        for batch in pbar:
            images = batch['image'].to(device)
            detection_labels = batch['detection'].to(device)
            localization_labels = batch['localization'].to(device)
            gan_labels = batch['gan'].to(device)
            
            outputs = model(images)
            loss, loss_dict = loss_fn(
                outputs, 
                detection_labels, 
                localization_labels, 
                gan_labels
            )
            
            total_loss += loss.item()
            for key in loss_dict_sum:
                loss_dict_sum[key] += loss_dict[key]
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(val_loader)
    for key in loss_dict_sum:
        loss_dict_sum[key] /= len(val_loader)
    
    return avg_loss, loss_dict_sum

print("✓ Training functions defined")

## 7. Train Model (Quick Validation - 5 Epochs, ~15 minutes)

In [ ]:
num_epochs = 5  # Quick validation (full training: 50+ epochs)
best_val_loss = float('inf')

print("\n" + "="*70)
print("Starting Training - Quick Validation (5 epochs)")
print("="*70 + "\n")

In [ ]:
# ============================================================================
# EXECUTE TRAINING LOOP
# ============================================================================

history = {
    'train_loss': [],
    'val_loss': [],
    'train_losses': {'detection': [], 'localization': [], 'gan': []},
    'val_losses': {'detection': [], 'localization': [], 'gan': []}
}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 70)
    
    # Train
    train_loss, train_loss_dict = train_epoch(model, train_loader, optimizer, loss_fn, device)
    history['train_loss'].append(train_loss)
    for key in train_loss_dict:
        history['train_losses'][key].append(train_loss_dict[key])
    
    # Validate
    val_loss, val_loss_dict = validate(model, val_loader, loss_fn, device)
    history['val_loss'].append(val_loss)
    for key in val_loss_dict:
        history['val_losses'][key].append(val_loss_dict[key])
    
    # Print metrics
    print(f"Train Loss: {train_loss:.4f}")
    print(f"  - Detection: {train_loss_dict['detection']:.4f}")
    print(f"  - Localization: {train_loss_dict['localization']:.4f}")
    print(f"  - GAN: {train_loss_dict['gan']:.4f}")
    
    print(f"Val Loss: {val_loss:.4f}")
    print(f"  - Detection: {val_loss_dict['detection']:.4f}")
    print(f"  - Localization: {val_loss_dict['localization']:.4f}")
    print(f"  - GAN: {val_loss_dict['gan']:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'best_model_epoch_{epoch+1}.pt')
        print(f"✓ Best model saved (val_loss: {val_loss:.4f})")
    
    # Learning rate scheduling
    scheduler.step()

print("\n" + "="*70)
print(f"✓ Training Complete! Best val loss: {best_val_loss:.4f}")
print("="*70)


## 8. Evaluation & Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_title('Total Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Detection loss
axes[0, 1].plot(history['train_losses']['detection'], label='Train', linewidth=2)
axes[0, 1].plot(history['val_losses']['detection'], label='Val', linewidth=2)
axes[0, 1].set_title('Detection Loss', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Localization loss
axes[1, 0].plot(history['train_losses']['localization'], label='Train', linewidth=2)
axes[1, 0].plot(history['val_losses']['localization'], label='Val', linewidth=2)
axes[1, 0].set_title('Localization Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# GAN classification loss
axes[1, 1].plot(history['train_losses']['gan'], label='Train', linewidth=2)
axes[1, 1].plot(history['val_losses']['gan'], label='Val', linewidth=2)
axes[1, 1].set_title('GAN Classification Loss', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training history plot saved")

## 9. Inference Example

In [ ]:
# Single inference example
model.eval()

with torch.no_grad():
    # Create a random sample
    sample_image = torch.randn(1, 3, 224, 224).to(device)
    
    # Forward pass
    outputs = model(sample_image)
    
    print("\nInference Example:")
    print(f"  Detection output shape: {outputs['detection'].shape}")
    print(f"  Detection logit: {outputs['detection'].item():.4f}")
    print(f"  Detection probability: {torch.sigmoid(outputs['detection']).item():.4f}")
    print(f"\n  Localization mask shape: {outputs['localization'].shape}")
    print(f"  GAN classification shape: {outputs['gan_classification'].shape}")
    print(f"  GAN class probabilities: {torch.sigmoid(outputs['gan_classification'])}")

## 10. Summary & Next Steps

✅ **Completed Stages:**
1. **Stage 1**: Adversarial Data Augmentation (FGSM/PGD setup)
2. **Stage 2**: Dual-Branch Feature Extraction (ResNet50 + ViT)
3. **Stage 3**: Adversarial Feature Fusion (Cross-attention)
4. **Stage 4**: Multi-Task Output Heads (Detection, Localization, GAN Classification)
5. **Stage 5**: XAI-Guided Training (Framework ready)

⏭️ **Next Steps for Full Implementation:**
- Replace dummy dataset with real MRI data
- Fine-tune loss weights using validation metrics
- Implement Grad-CAM++ visualization for Stage 5
- Add evaluation metrics (AUC, F1, IoU)
- When ready, add Stage 6 (PSO) for hyperparameter optimization